In [21]:
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    groq_api_key=groq_api_key
)

response = llm.invoke("rocket lanuched by india recently")
print(response.content)

As of my knowledge cutoff in 2023, some recent notable rocket launches by India include:

1. **SSLV-D1**: On August 7, 2022, India launched its new Small Satellite Launch Vehicle (SSLV) from the Satish Dhawan Space Centre in Sriharikota. Although the mission was not fully successful, it marked an important step in the development of India's small satellite launch capabilities.
2. **PSLV-C53**: On June 30, 2022, India launched the PSLV-C53 rocket, carrying three satellites from Singapore, from the Satish Dhawan Space Centre.
3. **GSLV-F10**: On August 12, 2021, India launched the GSLV-F10 rocket, carrying the EOS-03 satellite, from the Satish Dhawan Space Centre.
4. **PSLV-C51**: On February 28, 2021, India launched the PSLV-C51 rocket, carrying the Amazonia-1 satellite from Brazil and 18 other smaller satellites, from the Satish Dhawan Space Centre.

Please note that my knowledge may not be up-to-date, and there may have been more recent launches that I am not aware of. For the latest 

In [2]:
from langchain_community.document_loaders import WebBaseLoader
loader=WebBaseLoader('https://www.anthropic.com/careers')
page_data=loader.load().pop().page_content
print(page_data)

C:\Users\Kasthur Kumar\AppData\Local\Temp\ipykernel_23176\3287256658.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


Careers \ AnthropicSkip to main contentSkip to footerResearchPolicyCommitmentsLearnNewsTry ClaudeShape how AI meets the worldAnthropic builds Claude—AI designed to be helpful, honest, and harmless. We're researchers, engineers, and builders from a range of disciplines, working to make sure powerful AI goes well for everyone. If you're drawn to hard problems with real stakes, we'd like to meet you.Explore open rolesBuilding AnthropicOur co-founders discuss the origins of Anthropic, the “race to the top” in AI development, and where AI technology will go from here.Principles that guide how we show up for each other and our missionAct for the global goodWe strive to make decisions that maximize positive outcomes for humanity in the long run. This means we’re willing to be very bold in the actions we take to ensure our technology is a robustly positive force for good. We take seriously the task of safely guiding the world through a technological revolution that has the potential to change 

In [3]:
from langchain_core.prompts import PromptTemplate
prompt_extract=PromptTemplate.from_template(
    """
Minimum education: Bachelor’s degree or an equivalent combination of education, training, and/or experience

Required field of study: A field relevant to the role as demonstrated through coursework, training, or professional experience

Minimum years of experience: Years of experience required will correlate with the internal job level requirements for the position

Location-based hybrid policy: Currently, we expect all staff to be in one of our offices at least 25% of the time. However, some roles may require more time in our offices.

Visa sponsorship: We do sponsor visas! However, we aren't able to successfully sponsor visas for every role and every candidate. But if we make you an offer, we will make every reasonable effort to get you a visa, and we retain an immigration lawyer to help with this.
json with no premeble'

"""
)
chain_extract=prompt_extract | llm
res=chain_extract.invoke(input={'page_data':page_data})
print(res.content)


```
{
  "minimum_education": "Bachelor's degree",
  "equivalent_experience": "an equivalent combination of education, training, and/or experience",
  "required_field_of_study": "A field relevant to the role",
  "minimum_years_of_experience": "Years of experience required will correlate with the internal job level requirements for the position",
  "location_based_hybrid_policy": "at least 25% of the time in one of our offices",
  "visa_sponsorship": "We sponsor visas, but with varying success rates"
}
```


In [4]:
from langchain_core.output_parsers import JsonOutputParser
json_parser=JsonOutputParser()
json_res=json_parser.parse(res.content)
json_res

{'minimum_education': "Bachelor's degree",
 'equivalent_experience': 'an equivalent combination of education, training, and/or experience',
 'required_field_of_study': 'A field relevant to the role',
 'minimum_years_of_experience': 'Years of experience required will correlate with the internal job level requirements for the position',
 'location_based_hybrid_policy': 'at least 25% of the time in one of our offices',
 'visa_sponsorship': 'We sponsor visas, but with varying success rates'}

In [5]:
type(json_res)

dict

In [6]:
import pandas as pd
df=pd.read_csv("my_portfolio.csv")


In [9]:
import chromadb
import uuid

client = chromadb.PersistentClient("vectorstore")
collection = client.get_or_create_collection(name="portfolio")

if collection.count() == 0:
    for _, row in df.iterrows():
        collection.add(
            documents=[row["Techstack"]],
            metadatas=[{"links": row["Links"]}],
            ids=[str(uuid.uuid4())]
        )

In [13]:
links =collection.query(query_texts=job['minimum_education'],n_results=2).get('metadatas',[])
links

[[{'links': 'https://example.com/react-portfolio'},
  {'links': 'https://example.com/react-native-portfolio'}]]

In [12]:
job =json_res
job['minimum_education']

"Bachelor's degree"

In [14]:
prompt_email = PromptTemplate.from_template(
        """
        ### JOB DESCRIPTION:
        {job_description}
        
        ### INSTRUCTION:
        You are Mohan, a business development executive at AtliQ. AtliQ is an AI & Software Consulting company dedicated to facilitating
        the seamless integration of business processes through automated tools. 
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability, 
        process optimization, cost reduction, and heightened overall efficiency. 
        Your job is to write a cold email to the client regarding the job mentioned above describing the capability of AtliQ 
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase Atliq's portfolio: {link_list}
        Remember you are Mohan, BDE at AtliQ. 
        Do not provide a preamble.
        ### EMAIL (NO PREAMBLE):
        
        """
        )

chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res.content)

Subject: Expert AI & Software Consulting Services for Seamless Business Integration

Dear Hiring Manager,

I came across your job description and was impressed by the role's requirements. As a Business Development Executive at AtliQ, I'd like to introduce our company as a potential partner in fulfilling your needs. AtliQ is an AI & Software Consulting company dedicated to facilitating the seamless integration of business processes through automated tools.

With our expertise, we can empower your enterprise with tailored solutions, fostering scalability, process optimization, cost reduction, and heightened overall efficiency. Our team is well-versed in developing innovative solutions that cater to specific business requirements. We believe our capabilities align with your job requirements, including the need for a strong foundation in a relevant field of study and a minimum of a Bachelor's degree.

Our portfolio showcases our expertise in React and React Native, with notable projects fe